# set_methods
Sets are for uniqueness and fast membership. Their methods model real engineering tasks cleanly: deduplication, allowlists, blocklists, overlap analysis, capability comparisons, and fast seen-item tracking.

## 1. Problem First
Why does this matter in real systems?
- Large membership checks should not scan lists repeatedly.
- Duplicate removal is common in event and identity pipelines.
- Set algebra maps naturally to overlap and missing-coverage questions.

In [ ]:
blocked = {"bad-ip", "evil-ip"}
blocked.add("spam-ip")
blocked.update(["bot-ip", "scan-ip"])
print(blocked)

## 2. Minimal Concept
Key set operations and ideas:
- Methods: `add()`, `update()`, `remove()`, `discard()`, `pop()`, `clear()`, `copy()`
- Algebra: `union()`, `intersection()`, `difference()`, `symmetric_difference()`
- Relations: `issubset()`, `issuperset()`, `isdisjoint()`
- Variants: mutable `set`, immutable `frozenset`, set comprehension

## 3. Mental Model
How Python actually behaves
Sets are hash-table-based collections of unique hashable elements. They optimize for membership and uniqueness, not indexing or ordering. Set methods either mutate the current set or produce new sets depending on which API you call.

In [ ]:
values = [1, 2, 2, 3, 3]
unique = set(values)
print(unique)
print(2 in unique)
copy_unique = unique.copy()
copy_unique.add(99)
print(unique)
print(copy_unique)

## 4. Internal Mechanics
Membership relies on hashing, which is why mutable unhashable values cannot be elements. Some methods mutate in place, while algebraic methods often return new set objects. `frozenset` exists when you need an immutable hashable set-like value.

In [ ]:
import dis

def is_blocked(blocked, item):
    return item in blocked

dis.dis(is_blocked)
a = {1, 2, 3}
b = {3, 4, 5}
print(a.union(b))
print(a.intersection(b))
print(a.difference(b))
print(a.symmetric_difference(b))
frozen = frozenset([1, 2, 3])
print(frozen)

## 5. Edge Cases
Where it breaks:
- `remove()` raises `KeyError`, while `discard()` does not.
- `pop()` removes an arbitrary element, not a predictable one.
- Converting to a set destroys duplicates and any meaningful order.
- Mutable values like lists and dicts cannot be inserted.

In [ ]:
sample = {1, 2, 3}
sample.discard(99)
print(sample)
try:
    sample.remove(99)
except KeyError as exc:
    print(type(exc).__name__)
print(sample.pop())
try:
    bad = {[1, 2]}
except TypeError as exc:
    print(type(exc).__name__)

## 6. Performance Thinking
- Average membership is O(1).
- `add()` and `discard()` are O(1) average.
- Algebraic operations scale with input sizes.
- Sets are usually a performance win when repeated membership dominates.

## 7. Real Use Case
Sets are ideal for duplicate removal, fast allowlist checks, and mathematical overlap analysis between system capabilities.

In [ ]:
raw_ids = [101, 102, 101, 103, 102]
unique_ids = set(raw_ids)
print(unique_ids)
team_a = {"read", "write", "deploy"}
team_b = {"read", "metrics"}
print(team_b.issubset(team_a))
print(team_a.issuperset(team_b))
print(team_a.isdisjoint({"admin"}))
squares = {x * x for x in range(5)}
print(squares)
print(max(unique_ids), min(unique_ids), len(unique_ids), sorted(unique_ids))

## 8. Anti-Pattern
What beginners do wrong:
- Use sets where order or duplicate counts actually matter.
- Depend on `pop()` returning a specific value.
- Treat `remove()` and `discard()` as interchangeable.
- Forget that `frozenset` is the immutable variant.

In [ ]:
events = ["ERROR", "ERROR", "WARN"]
print(set(events))
print("frequency information is gone")

## 9. Interview Signals
Questions FAANG asks:
- Why are set lookups usually O(1) average?
- What is the difference between `remove()` and `discard()`?
- When should duplicates prevent you from using a set?
- What is `frozenset` for?

## 10. Exercise (Non-trivial)
Design an event-ingestion guardrail using sets. It should remove duplicates, block blacklisted actors, compare requested capabilities against allowed capabilities, and explain where a set is perfect and where another structure is needed for counts or order.

In [ ]:
def filter_requests(requests, blocked_ids):
    seen = set()
    allowed = []
    for request in requests:
        request_id = request["id"]
        if request_id in blocked_ids or request_id in seen:
            continue
        seen.add(request_id)
        allowed.append(request)
    return allowed

# Task:
# 1. Add capability-overlap checks with set algebra.
# 2. Explain why set is faster than list membership here.
# 3. Describe what extra structure would preserve counts.